In [1]:
%load_ext pyspecdata.ipy
import numpy as np
import scipy
from scipy.optimize import root_scalar

Here we take the aggregation number vs. $w_0$, and use
it to determine what w0 we want to get a
particular water loading.
There are many function definitions that
all come together at the end.

Start by defining a function to determine the
aggregation number using equation taken
from the RM 2H paper.
using a dictionary here is a bit
round-about, but it's because of how the
original code was written

In [28]:
params = {}
params['k'] = 0.174
params['zeroth'] = 15.1
params['first_b'] = 0.260
params['first_m'] = 7.15
params['second_l'] = 0.673
params['second_c'] = -37.1
def aggregation_number(params, w0):
    # exp(βn) = exp(βa) + exp(β(mx+b)) + exp(β(lx²+c))
    k = params['k']
    model = np.exp(k*params['zeroth'])
    model += np.exp(k*(params['first_m']*w0 + params['first_b']))
    model += np.exp(k*(params['second_l']*w0**2 + params['second_c']))
    model = np.log(model)/k
    # when infinite, assume the quadratic term dominates
    mask = ~np.isfinite(model)
    if np.isscalar(mask):
        if mask:
            model = params['second_l']*w0**2 + params['second_c']
            return w0*model
    elif any(mask):
        model[mask] = params['second_l']*w0[mask]**2 + params['second_c']
    return model

function to determine the weight of the water (in Da)
from the w0

In [29]:
def water_weight(w0):
    return 18*w0*aggregation_number(params,w0)

equations from Pal2011FluCorSpe for
determining the radius -- they define two
equations for two regimes

In [30]:
def smaller_radius(w):
    return 0.175*w+1.5
def larger_radius(w):
    delta = 1.5
    gamma = 0.043
    y = 20.4
    numerator = y*(1+2*gamma)
    radius = 3*delta*w/numerator*(
            1+numerator/3/w
            -numerator**2/9/w**2
            +5*numerator**3/81/w**3)
    return radius
@np.vectorize
def rm_radius(w):
    return smaller_radius(w) if w<20 else larger_radius(w)

These are the actual functions that you use
to get all the info about an RM, specified
either by $w_0$ or by weight

In [31]:
def rm_by_w0(thisw0):
    nbar = aggregation_number(params,thisw0)
    ww = water_weight(thisw0)/1e3
    nw = thisw0*nbar
    assert np.isclose(ww*1e3,18*nw), str(ww*1e3)+" doesn't match "+str(18*nw)
    print("at",thisw0,"\n",
          "number of aot is",nbar,"\n",
          "number of waters is",nw,"\n",
          "water weight is",ww,'kDa',"\n",
          "concentration needed for single loading",55.44/nw/1e-3,'mM',"\n",
          "diameter",2*rm_radius(thisw0),'nm',"\n",
          "diameter of water",2*rm_radius(thisw0)-3.0,'nm',"\n",
          )
def rm_by_weight(thisweight):
    result = root_scalar(
            lambda x: water_weight(x)-thisweight,
            x0=0,
            x1=thisweight/10)
    assert result.flag == 'converged'
    rounded = np.round(result.root*10)/10
    print("I find a weight of",thisweight,"at w0=",result.root,"\n",
            "confirming ", water_weight(result.root),"\n",
            "with rounding to",rounded,
            water_weight(
                rounded
                ))

In [37]:
rm_by_w0(3)

at 3 
 number of aot is 23.29117979004874 
 number of waters is 69.87353937014622 
 water weight is 1.2577237086626318 kDa 
 concentration needed for single loading 793.4334012524201 mM 
 diameter 4.05 nm 
 diameter of water 1.0499999999999998 nm 



In [39]:
# rm_by_weight(10e3)

In [41]:
# Manuscript
# isooctane
# for w0=3
# Temperature 298.15 K (high)
# viscosity of isooctane at 298.15 K is 0.473 mPas 

eta_isooctane_rt = 0.473e-3
T = 298.15
r = 4.05e-9/2
k= scipy.constants.k
tau = 4*pi*eta_isooctane_rt*r**3/(3*k_B*T)
print(f"correlation time {tau/1e-12:0.2f} ps and correction factor {log10(293/250)} and RM radius is {r/1e-9:0.2f} nm")

correlation time 3996.75 ps and correction factor 0.06892761168207182 and RM radius is 2.02 nm


In [42]:
rm_by_w0(20)

at 20 
 number of aot is 232.10000111187802 
 number of waters is 4642.00002223756 
 water weight is 83.55600040027608 kDa 
 concentration needed for single loading 11.9431279048716 mM 
 diameter 10.698767768078 nm 
 diameter of water 7.698767768078 nm 



In [44]:
# Manuscript
# isooctane
# for w0=20
# Temperature 298.15 K (high)
# viscosity of isooctane at 298.15 K is 0.473 mPas 

eta_isooctane_rt = 0.473e-3
T = 298.15
r = 10.698767768078e-9/2
k= scipy.constants.k
tau = 4*pi*eta_isooctane_rt*r**3/(3*k_B*T)
print(f"correlation time {tau/1e-12:0.2f} ps and correction factor {log10(293/250)} and RM radius is {r/1e-9:0.2f} nm")

correlation time 73678.88 ps and correction factor 0.06892761168207182 and RM radius is 5.35 nm
